In [13]:
%cd /Users/akarsh/Documents/projects/fastInfer/StoryForge

/Users/akarsh/Documents/projects/fastInfer/StoryForge


In [14]:
import numpy as np
import torch

from storyforge.config import load_config
from storyforge.batching import get_batch
from tokenizers import Tokenizer
from torch import nn

In [15]:
config = load_config("configs/default.toml")

In [16]:
train_tokens = np.load(
    config["encoding"]["train_path"],
    mmap_mode="r",
)

In [17]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [18]:
rng = np.random.default_rng(config["seed"])

In [19]:
inputs, targets = get_batch(
    tokens=train_tokens,
    batch_size=config["training"]["batch_size"],
    context=config["model"]["max_context"],
    rng=rng,
    device=device,
)


In [20]:
print("Inputs:", inputs.shape)
print("Targets:", targets.shape)
print("Integer type:", inputs.dtype)
print("Device:", inputs.device)

Inputs: torch.Size([8, 256])
Targets: torch.Size([8, 256])
Integer type: torch.int64
Device: mps:0


In [21]:
assert torch.equal(inputs[:, 1:], targets[:, :-1])

In [22]:
tokenizer = Tokenizer.from_file(
    config["tokenizer"]["path"]
)


In [23]:
vocab_size = tokenizer.get_vocab_size()
d_model = config["model"]["d_model"]
max_context = config["model"]["max_context"]

In [24]:
token_embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=d_model,
).to(device)

In [25]:
position_embedding = nn.Embedding(
    num_embeddings=max_context,
    embedding_dim=d_model,
).to(device)

In [34]:
inputs.shape

torch.Size([8, 256])

In [26]:
token_vectors = token_embedding(inputs)

In [27]:
print(token_vectors.shape)

torch.Size([8, 256, 384])


In [28]:
sequence_length = inputs.shape[1]

In [29]:
positions = torch.arange(
    sequence_length,
    device=device,
)

In [30]:
position_vectors = position_embedding(positions)

In [33]:
position_vectors.shape

torch.Size([256, 384])

In [31]:
hidden = token_vectors + position_vectors

In [37]:
hidden.shape

torch.Size([8, 256, 384])